# Publication plot: 2D fit distribution

Expected selected-event distribution in the same reconstructed $(\log_{10} Q^2, p_n)$ bins used by `uboone_numuCC1p_zexp_2D.xml`. The figure is displayed only; no file is written.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import uproot
from matplotlib.colors import LogNorm

mpl.rcParams.update({
    "figure.dpi": 140,
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.linewidth": 1.1,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "legend.frameon": False,
    "mathtext.fontset": "stix",
})

In [ ]:
# Inputs and binning copied from the XML fit configuration.
#
# Both datasets are produced by this notebook from the same code: the NuWro fake-data
# figures first, the open-data ones at the end. Everything that differs between them
# lives in SUITES; the functions below take one of these dicts and are otherwise
# identical for the two.
ROOT_FILE = Path("/nevis/riverside/data/epelaez/ngem/intermediate_files/minimal_withspline_df.root")
TREE_NAME = "tree"
FIT_RESULTS_ROOT = Path("/nevis/hopper/data/epelaez/axial_mass")

# The prediction is normalized by net_weight, which create_df builds with the
# open-data weighting config, i.e. to the total beam-on POT. That is the number
# PROfit is told in <MCFile pot=...>, so it has to match or the stack and the
# PROfit error band would disagree by the ratio. 9.3221e19 is the run 1/3/4a/4b
# open-data sum (pot_dic.csv); it was 9.57e19 before the run 4a POT correction.
MC_POT = 9.3221e19

Q2_LOG_EDGES = np.array([-2.00, -1.50, -1.20, -1.00, -0.85, -0.70, -0.55, -0.40, -0.20, 0.20])
PN_EDGES = np.array([0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 1.00])

BRANCHES = [
    "afro_1mu1p_Q2", "afro_1mu1p_Pn", "afro_1mu1p_sel", "net_weight", "net_weight_nuwro",
    "isdata", "isext", "isdirt", "isnuwro", "wc_truth_nuScatType", "GTruth_gQ2",
]

search_root = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (search_root, *search_root.parents)
     if (path / 'python' / 'notebooks').is_dir() and (path / 'figs').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate the ma_zexp repository root')
FIGURE_ROOT = REPO_ROOT / 'figs'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)


def _suite(suite, fit, data_label, target_pot, data_mask, data_weight, header, suffix):
    directory = FIT_RESULTS_ROOT / f"{suite}_fit_results" / fit
    return dict(
        suite=suite, fit=fit, data_label=data_label, target_pot=target_pot,
        data_mask=data_mask, data_weight=data_weight, header=header, suffix=suffix,
        profit_plot_file=directory / f"{fit}_v1_PROplot.root",
        pot_scale=target_pot / MC_POT,
    )


SUITES = {
    # NuWro keeps the unsuffixed filenames that paper/body.tex includes.
    # NuWro is excluded from the open-data weighting, so it carries its own weight branch.
    "nuwro": _suite("nuwro", "ma", "NuWro", 1.75e21,
                    ("isnuwro",), "net_weight_nuwro", "Simulation", ""),
    # Real beam-on data carries net_weight, the same branch and weighting the opendata
    # XML hands PROfit, so the points and the stack agree by construction.
    "opendata": _suite("opendata", "ma", "Open data", MC_POT,
                       ("isdata",), "net_weight", "Preliminary", "_opendata"),
}

for _name, _cfg in SUITES.items():
    if not _cfg["profit_plot_file"].is_file():
        raise FileNotFoundError(
            f"{_cfg['profit_plot_file']} does not exist; run xml/run_all.sh first")
    print(f"{_name:<9} data={_cfg['data_label']:<10} target POT={_cfg['target_pot']:.4e}  "
          f"POT scale={_cfg['pot_scale']:.4f}")


In [ ]:
def prepare_suite(cfg):
    """Load the tree and bin the prediction and the data overlay for one suite."""
    POT_SCALE = cfg["pot_scale"]
    DATA_LABEL = cfg["data_label"]

    # Read only the columns needed for this plot.
    with uproot.open(ROOT_FILE) as root_file:
        arrays = root_file[TREE_NAME].arrays(BRANCHES, library="np")

    # Union of the overlay, dirt, and EXT selections in the XML.
    overlay = (~arrays["isdata"] & ~arrays["isext"] & ~arrays["isdirt"] &
               ~arrays["isnuwro"] & arrays["afro_1mu1p_sel"])
    dirt = (~arrays["isdata"] & ~arrays["isext"] & arrays["isdirt"] &
            arrays["afro_1mu1p_sel"])
    ext = (~arrays["isdata"] & arrays["isext"] & ~arrays["isdirt"] &
           arrays["afro_1mu1p_sel"])
    selected = overlay | dirt | ext

    # True-Q^2 is defined for simulated neutrino interactions, so use the
    # selected overlay component (not data-driven EXT or dirt backgrounds).
    selected_true_q2 = arrays["GTruth_gQ2"][overlay]
    selected_true_q2_weights = arrays["net_weight"][overlay] * POT_SCALE
    true_q2_finite = (np.isfinite(selected_true_q2) &
                      np.isfinite(selected_true_q2_weights) &
                      (selected_true_q2 > 0))
    selected_true_q2 = selected_true_q2[true_q2_finite]
    selected_true_q2_weights = selected_true_q2_weights[true_q2_finite]

    q2 = arrays["afro_1mu1p_Q2"][selected]
    pn = arrays["afro_1mu1p_Pn"][selected]
    weights = arrays["net_weight"][selected] * POT_SCALE
    interaction_type = arrays["wc_truth_nuScatType"][selected]

    finite = np.isfinite(q2) & np.isfinite(pn) & np.isfinite(weights) & (q2 > 0)
    log_q2 = np.log10(q2[finite])
    pn = pn[finite]
    weights = weights[finite]
    interaction_type = interaction_type[finite]

    fit_counts, _, _ = np.histogram2d(
        log_q2, pn, bins=(Q2_LOG_EDGES, PN_EDGES), weights=weights
    )

    # Data overlay, per SUITE_CONFIG. NuWro fake data carries net_weight_nuwro, already
    # normalized to 1.75e21 POT; real beam-on data carries net_weight, the same branch and
    # weighting the opendata XML gives PROfit, so the two agree by construction.
    data_selected = arrays["afro_1mu1p_sel"].copy()
    for _flag in cfg["data_mask"]:
        data_selected &= arrays[_flag]
    data_q2 = arrays["afro_1mu1p_Q2"][data_selected]
    data_pn = arrays["afro_1mu1p_Pn"][data_selected]
    data_weights = arrays[cfg["data_weight"]][data_selected]
    data_finite = (
        np.isfinite(data_q2) & np.isfinite(data_pn) &
        np.isfinite(data_weights) & (data_q2 > 0)
    )
    data_log_q2 = np.log10(data_q2[data_finite])
    data_pn = data_pn[data_finite]
    data_weights = data_weights[data_finite]
    data_counts, _, _ = np.histogram2d(
        data_log_q2, data_pn, bins=(Q2_LOG_EDGES, PN_EDGES), weights=data_weights
    )
    data_sumw2, _, _ = np.histogram2d(
        data_log_q2, data_pn, bins=(Q2_LOG_EDGES, PN_EDGES), weights=data_weights**2
    )

    print(f"Selected entries: {finite.sum():,}")
    print(f"Expected yield inside fit range: {fit_counts.sum():,.1f} events")
    print(f"{DATA_LABEL} yield inside fit range: {data_counts.sum():,.1f} events")
    return dict(
        cfg=cfg, fit_counts=fit_counts, log_q2=log_q2, pn=pn, weights=weights,
        interaction_type=interaction_type, data_counts=data_counts,
        data_sumw2=data_sumw2, selected_true_q2=selected_true_q2,
        selected_true_q2_weights=selected_true_q2_weights,
    )


In [ ]:
def plot_occupancy(st):
    """2D occupancy of the fit binning (not saved; format left to the user)."""
    fit_counts = st["fit_counts"]
    TARGET_POT = st["cfg"]["target_pot"]

    positive = fit_counts[fit_counts > 0]
    norm = LogNorm(vmin=max(positive.min(), 1.0), vmax=positive.max())

    fig, ax = plt.subplots(figsize=(8.0, 6.2), constrained_layout=True)
    mesh = ax.pcolormesh(
        Q2_LOG_EDGES,
        PN_EDGES,
        fit_counts.T,
        cmap="magma",
        norm=norm,
        shading="flat",
        edgecolors=(1, 1, 1, 0.32),
        linewidth=0.7,
        rasterized=True,
    )

    cbar = fig.colorbar(mesh, ax=ax, pad=0.025, aspect=32)
    cbar.set_label("Expected events / fit bin", labelpad=12)
    cbar.ax.tick_params(which="both", direction="in")

    ax.set_xlabel(r"$\log_{10}\!\left(Q^2_{\mathrm{reco}}/\mathrm{GeV}^2\right)$", labelpad=8)
    ax.set_ylabel(r"$p_n^{\mathrm{reco}}$ [GeV/$c$]", labelpad=8)
    ax.set_xlim(Q2_LOG_EDGES[[0, -1]])
    ax.set_ylim(PN_EDGES[[0, -1]])
    ax.set_xticks(Q2_LOG_EDGES)
    ax.set_yticks(PN_EDGES)
    ax.tick_params(axis="x", labelrotation=35)

    ax.text(
        0.025, 0.975, r"$\bf{MicroBooNE}$  Preliminary",
        transform=ax.transAxes, ha="left", va="top", color="white", fontsize=13,
    )
    ax.text(
        0.975, 0.975,
        rf"$\nu_\mu$ CC1p selection\n$\mathrm{{POT}}={TARGET_POT:.1e}$",
        transform=ax.transAxes, ha="right", va="top", color="white", fontsize=11,
    )

    # Intentionally no savefig: leave saving and final format choices to the user.
    plt.show()

In [ ]:
def plot_slices(st):
    """Per-p_n-slice Q^2 spectra; saves selection_slices<suffix>.pdf."""
    cfg = st["cfg"]
    interaction_type, log_q2, pn = st["interaction_type"], st["log_q2"], st["pn"]
    weights, fit_counts = st["weights"], st["fit_counts"]
    data_counts, data_sumw2 = st["data_counts"], st["data_sumw2"]
    PROFIT_PLOT_FILE = cfg["profit_plot_file"]
    DATA_LABEL, TARGET_POT = cfg["data_label"], cfg["target_pot"]
    HEADER_LABEL, FIG_SUFFIX = cfg["header"], cfg["suffix"]

    # Generator interaction codes follow wc_truth_nuScatType.
    interaction_specs = [
        ("QE",  interaction_type == 1,  "#3070AD"),
        ("MEC", interaction_type == 10, "#C66526"),
        ("RES", interaction_type == 4,  "#469C76"),
        ("DIS", interaction_type == 3,  "#DCA237"),
        ("Other", ~np.isin(interaction_type, [1, 10, 4, 3]), "#9AA4B2"),
    ]

    interaction_counts = []
    for _, mask, _ in interaction_specs:
        counts, _, _ = np.histogram2d(
            log_q2[mask], pn[mask], bins=(Q2_LOG_EDGES, PN_EDGES), weights=weights[mask]
        )
        interaction_counts.append(counts)
    interaction_counts = np.asarray(interaction_counts)

    # PROfit's pre-fit error-band graphs are the canonical materialization of
    # PROfit_syst.bin + PROfit_prop.bin + PROfit_detvar_props.bin.  Read the
    # eight p_n slices directly instead of reproducing the covariance machinery.
    with uproot.open(PROFIT_PLOT_FILE) as profit_plot_file:
        total_prediction_uncertainty = np.column_stack([
            profit_plot_file[
                f"ErrorBand/Var0/nu_uBooNE_numuCC1p_preerrband_slice_ybin{pn_bin + 1}"
            ].member("fEYhigh")
            for pn_bin in range(len(PN_EDGES) - 1)
        ])
        # Total pre-fit fractional covariance of the collapsed 2D spectrum and the
        # central value PROfit built it from.  PROfit flattens the (Q^2, p_n) grid
        # with Q^2 as the slow index: flat_bin = q2_bin * n_pn_bins + pn_bin.
        profit_fractional_covariance = profit_plot_file[
            "Covariance/collapsed_total_frac_cov"
        ].values()
        profit_central_value = profit_plot_file[
            "ErrorBand/Var0/nu_uBooNE_numuCC1p_cv2d"
        ].values()

    if total_prediction_uncertainty.shape != fit_counts.shape:
        raise ValueError(
            f"PROfit uncertainty shape {total_prediction_uncertainty.shape} does not match "
            f"the selection histogram shape {fit_counts.shape}"
        )

    # Absolute systematic covariance (MC stat included) in PROfit's flat bin order.
    # It is the same total uncertainty as the drawn band, so the chi^2 below is
    # computed against exactly what the figure shows.
    profit_central_value_flat = profit_central_value.reshape(-1)
    prediction_covariance = profit_fractional_covariance * np.outer(
        profit_central_value_flat, profit_central_value_flat
    )
    if not np.allclose(
        np.sqrt(np.diag(prediction_covariance)).reshape(fit_counts.shape),
        total_prediction_uncertainty, rtol=0.15,
    ):
        raise ValueError(
            "The collapsed covariance diagonal does not reproduce the PROfit error "
            "bands; check the flat bin ordering"
        )


    def chi_square(residual, covariance):
        """Neyman chi^2 (PROfit's default): covariance already holds the data statistical variance."""
        return float(residual @ np.linalg.solve(covariance, residual))

    category_yields = interaction_counts.sum(axis=(1, 2))
    category_fractions = category_yields / category_yields.sum()
    prediction_upper = interaction_counts.sum(axis=0) + total_prediction_uncertainty
    data_upper = data_counts + np.sqrt(data_sumw2)
    panel_upper = np.maximum(prediction_upper, data_upper).max(axis=0)
    # Headroom keeps the hatched band clear of the two-column legend.
    row_ymax = [
        max(1.0, 1.70 * panel_upper[:4].max()),
        max(1.0, 1.70 * panel_upper[4:].max()),
    ]

    background = "#FFFFFF"
    foreground = "#303642"
    accent = "#020202"
    band_facecolor = mpl.colors.to_rgba(foreground, 0.30)
    band_hatchcolor = mpl.colors.to_rgba(foreground, 0.55)
    band_hatch = "/////"
    bin_centers = 0.5 * (Q2_LOG_EDGES[:-1] + Q2_LOG_EDGES[1:])
    bin_widths = np.diff(Q2_LOG_EDGES)

    with mpl.rc_context({
        "font.family": "sans-serif", "mathtext.fontset": "dejavusans", "hatch.linewidth": 0.5,
    }):
        fig, axes = plt.subplots(
            2, 4, figsize=(13.0, 7.6), sharex=True, sharey="row",
            facecolor=background,
        )

        for pn_bin, ax in enumerate(axes.flat):
            ax.set_facecolor(background)
            bottom = np.zeros(len(Q2_LOG_EDGES) - 1)

            for category, (label, _, color) in enumerate(interaction_specs):
                values = interaction_counts[category, :, pn_bin]
                ax.stairs(
                    bottom + values, Q2_LOG_EDGES, baseline=bottom, fill=True,
                    color=color, edgecolor="none", linewidth=0,
                    antialiased=False, label=label,
                )
                bottom += values

            prediction_uncertainty = total_prediction_uncertainty[:, pn_bin]
            band_lower = np.clip(bottom - prediction_uncertainty, 0, None)
            band_upper = bottom + prediction_uncertainty
            ax.fill_between(
                Q2_LOG_EDGES,
                np.r_[band_lower, band_lower[-1]],
                np.r_[band_upper, band_upper[-1]],
                step="post", facecolor=band_facecolor, edgecolor=band_hatchcolor,
                hatch=band_hatch, linewidth=0, zorder=8,
            )
            for band_edge in (band_lower, band_upper):
                ax.stairs(
                    band_edge, Q2_LOG_EDGES, baseline=None, color=foreground,
                    linestyle="--", linewidth=1.0, zorder=9,
                )

            ax.errorbar(
                bin_centers, data_counts[:, pn_bin],
                xerr=0.5 * bin_widths, yerr=np.sqrt(data_sumw2[:, pn_bin]),
                fmt="o", color="black", markerfacecolor="black",
                markersize=3.8, elinewidth=1.0, capsize=2.0,
                linestyle="none", zorder=10,
            )

            ax.text(
                0.95, 0.95,
                rf"${PN_EDGES[pn_bin]:.1f} \leq p_n^\mathrm{{reco}} < {PN_EDGES[pn_bin + 1]:.1f}$ GeV/$c$",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=10, color=foreground,
            )

            # Residual against the drawn stack; covariance is the sub-block of the
            # total pre-fit covariance for this p_n slice plus the data variance.
            slice_bins = np.arange(len(Q2_LOG_EDGES) - 1) * (len(PN_EDGES) - 1) + pn_bin
            slice_covariance = (
                prediction_covariance[np.ix_(slice_bins, slice_bins)]
                + np.diag(data_sumw2[:, pn_bin])
            )
            slice_chi2 = chi_square(data_counts[:, pn_bin] - bottom, slice_covariance)
            ax.set_xlim(Q2_LOG_EDGES[[0, -1]])
            ax.set_ylim(0, row_ymax[pn_bin // 4])
            ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=5, integer=True))
            ax.tick_params(
                which="both", direction="in", top=True, right=True,
                colors=foreground, labelcolor=foreground, length=6,
            )
            for spine in ax.spines.values():
                spine.set_color(foreground)
                spine.set_linewidth(1.0)

            slice_yields = interaction_counts[:, :, pn_bin].sum(axis=1)
            slice_fractions = slice_yields / slice_yields.sum()
            handles = [mpl.patches.Patch(color=color) for _, _, color in interaction_specs]
            handles.append(mpl.patches.Patch(
                facecolor=band_facecolor, edgecolor=foreground, hatch=band_hatch,
                linestyle="--", linewidth=1.0,
            ))
            handles.append(mpl.lines.Line2D(
                [], [], color="black", marker="o", linestyle="none", markersize=4.0
            ))
            # Blank handle: the chi^2 fills the empty legend slot below the data marker.
            handles.append(mpl.patches.Patch(facecolor="none", edgecolor="none"))
            labels = [
                rf"{label} ({fraction:.1%})"
                for (label, _, _), fraction in zip(interaction_specs, slice_fractions)
            ]
            labels.extend([
                "Total unc.", DATA_LABEL,
                rf"$\chi^2/\mathrm{{N}} = {slice_chi2:.1f}/{len(slice_bins)}$",
            ])
            legend = ax.legend(
                handles, labels,
                loc="upper left", bbox_to_anchor=(0.02, 0.90),
                ncol=2, frameon=False, fontsize=10, labelspacing=0.35, columnspacing=0.45,
                title_fontsize=1, handlelength=1.3, handletextpad=0.6, borderaxespad=0
            )
            for text in legend.get_texts():
                text.set_color(foreground)
            legend.get_title().set_color(foreground)

        fig.text(0.065, 0.915, "MicroBooNE", color=accent, fontsize=14, weight="bold", va="bottom")
        fig.text(0.170, 0.915, HEADER_LABEL, color=accent, fontsize=14, va="bottom")
        fig.text(0.975, 0.915, f"{TARGET_POT:.2e} POT".replace("+", ""), color=accent, fontsize=14, ha="right", va="bottom")
        fig.supxlabel(
            r"$\log_{10}\!\left(Q^2_{\mathrm{reco}}/\mathrm{GeV}^2\right)$",
            color=foreground, fontsize=16, y=0.025,
        )
        fig.supylabel("Events / bin", color=foreground, fontsize=16, x=0.01)
        fig.subplots_adjust(left=0.055, right=0.975, bottom=0.105, top=0.91, hspace=0.08, wspace=0.08)

        fig.savefig(
            FIGURE_ROOT / f"selection_slices{FIG_SUFFIX}.pdf",
            format="pdf",
            dpi=600,
            bbox_inches="tight",
            facecolor="white",
        )

        # Intentionally no savefig.
        plt.show()
    # carried to plot_projections so the two figures share one covariance and style
    st.update(
        interaction_specs=interaction_specs, interaction_counts=interaction_counts,
        prediction_covariance=prediction_covariance, chi_square=chi_square,
        total_prediction_uncertainty=total_prediction_uncertainty,
        background=background, foreground=foreground, accent=accent,
        band_facecolor=band_facecolor, band_hatchcolor=band_hatchcolor,
        band_hatch=band_hatch,
    )


In [ ]:
def plot_projections(st):
    """Fully projected Q^2 and p_n spectra; saves selection_projections<suffix>.pdf."""
    cfg = st["cfg"]
    fit_counts, interaction_counts = st["fit_counts"], st["interaction_counts"]
    interaction_specs = st["interaction_specs"]
    prediction_covariance, chi_square = st["prediction_covariance"], st["chi_square"]
    background, foreground, accent = st["background"], st["foreground"], st["accent"]
    band_facecolor, band_hatchcolor = st["band_facecolor"], st["band_hatchcolor"]
    band_hatch = st["band_hatch"]
    PROFIT_PLOT_FILE = cfg["profit_plot_file"]
    DATA_LABEL, TARGET_POT = cfg["data_label"], cfg["target_pot"]
    HEADER_LABEL, FIG_SUFFIX = cfg["header"], cfg["suffix"]

    # Fully projected spectra.  PROfit stores the data and correlated total
    # uncertainty after integrating the other axis, so the band is read directly;
    # only the chi^2 projects the full covariance onto each axis.  The stack
    # reuses the category histograms binned above.
    n_q2_bins, n_pn_bins = fit_counts.shape
    q2_projection = np.kron(np.eye(n_q2_bins), np.ones((1, n_pn_bins)))
    pn_projection = np.kron(np.ones((1, n_q2_bins)), np.eye(n_pn_bins))

    with uproot.open(PROFIT_PLOT_FILE) as profit_plot_file:
        projection_specs = [
            {
                "edges": Q2_LOG_EDGES,
                "counts": interaction_counts.sum(axis=2),
                "data": profit_plot_file["ErrorBand/Var0/nu_uBooNE_numuCC1p_data"].values(),
                "data_error": profit_plot_file["ErrorBand/Var0/nu_uBooNE_numuCC1p_data"].errors(),
                "uncertainty": profit_plot_file["ErrorBand/Var0/nu_uBooNE_numuCC1p_preerrband"].member("fEYhigh"),
                "xlabel": r"$\log_{10}\!\left(Q^2_{\mathrm{reco}}/\mathrm{GeV}^2\right)$",
                "projection": q2_projection,
            },
            {
                "edges": PN_EDGES,
                "counts": interaction_counts.sum(axis=1),
                "data": profit_plot_file["ErrorBand/Var0/nu_uBooNE_numuCC1p_data_y"].values(),
                "data_error": profit_plot_file["ErrorBand/Var0/nu_uBooNE_numuCC1p_data_y"].errors(),
                "uncertainty": profit_plot_file["ErrorBand/Var0/nu_uBooNE_numuCC1p_preerrband_y"].member("fEYhigh"),
                "xlabel": r"$p_n^{\mathrm{reco}}$ [GeV/$c$]",
                "projection": pn_projection,
            },
        ]

    with mpl.rc_context({
        "font.family": "sans-serif", "mathtext.fontset": "dejavusans", "hatch.linewidth": 0.5,
    }):
        fig, axes = plt.subplots(2, 1, figsize=(7.0, 10.0), sharey=False, facecolor=background)

        for panel_index, (ax, spec) in enumerate(zip(axes, projection_specs)):
            edges = spec["edges"]
            centers = 0.5 * (edges[:-1] + edges[1:])
            widths = np.diff(edges)
            bottom = np.zeros(len(edges) - 1)
            ax.set_facecolor(background)

            for category, (label, _, color) in enumerate(interaction_specs):
                values = spec["counts"][category]
                ax.stairs(
                    bottom + values, edges, baseline=bottom, fill=True,
                    color=color, edgecolor="none", linewidth=0, antialiased=False,
                )
                bottom += values

            band_lower = np.clip(bottom - spec["uncertainty"], 0, None)
            band_upper = bottom + spec["uncertainty"]
            ax.fill_between(
                edges, np.r_[band_lower, band_lower[-1]],
                np.r_[band_upper, band_upper[-1]], step="post",
                facecolor=band_facecolor, edgecolor=band_hatchcolor,
                hatch=band_hatch, linewidth=0, zorder=8,
            )
            for band_edge in (band_lower, band_upper):
                ax.stairs(
                    band_edge, edges, baseline=None, color=foreground,
                    linestyle="--", linewidth=1.0, zorder=9,
                )
            ax.errorbar(
                centers, spec["data"], xerr=0.5 * widths, yerr=spec["data_error"],
                fmt="o", color="black", markerfacecolor="black", markersize=3.8,
                elinewidth=1.0, capsize=2.0, linestyle="none", zorder=10,
            )

            projected_covariance = (
                spec["projection"] @ prediction_covariance @ spec["projection"].T
                + np.diag(spec["data_error"] ** 2)
            )
            panel_chi2 = chi_square(spec["data"] - bottom, projected_covariance)
            ax.text(
                0.975, 0.965,
                rf"$\chi^2/\mathrm{{N}} = {panel_chi2:.1f}/{len(bottom)}$",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=11, color=foreground,
            )

            category_yields = spec["counts"].sum(axis=1)
            category_fractions = category_yields / category_yields.sum()
            handles = [mpl.patches.Patch(color=color) for _, _, color in interaction_specs]
            handles.append(mpl.patches.Patch(
                facecolor=band_facecolor, edgecolor=foreground, hatch=band_hatch,
                linestyle="--", linewidth=1.0,
            ))
            handles.append(mpl.lines.Line2D(
                [], [], color="black", marker="o", linestyle="none", markersize=4.0,
            ))
            labels = [
                rf"{label} ({fraction:.1%})"
                for (label, _, _), fraction in zip(interaction_specs, category_fractions)
            ]
            labels.extend(["Total pred. unc.", DATA_LABEL])
            if panel_index == 0:
                legend = ax.legend(
                    handles, labels, loc="upper left", bbox_to_anchor=(0.025, 0.975),
                    ncol=1, frameon=False, fontsize=10, labelspacing=0.35,
                    columnspacing=0.8, handlelength=1.6, borderaxespad=0,
                )
                for text in legend.get_texts():
                    text.set_color(foreground)

            panel_upper = max(
                np.max(bottom + spec["uncertainty"]),
                np.max(spec["data"] + spec["data_error"]),
            )
            ax.set_xlim(edges[[0, -1]])
            ax.set_ylim(0, 1.10 * panel_upper)
            ax.set_xlabel(spec["xlabel"], color=foreground, fontsize=16, labelpad=5)
            ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=6, integer=True))
            ax.tick_params(
                which="both", direction="in", top=True, right=True,
                colors=foreground, labelcolor=foreground, length=6,
            )
            for spine in ax.spines.values():
                spine.set_color(foreground)
                spine.set_linewidth(1.0)

        fig.text(0.120, 0.915, "MicroBooNE", color=accent, fontsize=14, weight="bold", va="bottom")
        fig.text(0.315, 0.915, HEADER_LABEL, color=accent, fontsize=14, va="bottom")
        fig.text(0.975, 0.915, f"{TARGET_POT:.2e} POT".replace("+", ""), color=accent, fontsize=14, ha="right", va="bottom")
        fig.supylabel("Events / bin", color=foreground, fontsize=16, x=0.015)
        fig.subplots_adjust(left=0.11, right=0.975, bottom=0.09, top=0.91, hspace=0.25)
        fig.savefig(
            FIGURE_ROOT / f"selection_projections{FIG_SUFFIX}.pdf", dpi=600, bbox_inches="tight", facecolor="white", format="pdf",
        )
        plt.show()

## NuWro fake data

The figures `paper/body.tex` includes as `selection_slices.pdf` and `selection_projections.pdf`.


In [ ]:
nuwro = prepare_suite(SUITES["nuwro"])
plot_occupancy(nuwro)
plot_slices(nuwro)
plot_projections(nuwro)

# The axial form-factor prior comparison below draws the selected MC true-Q^2
# distribution as a silhouette. It is normalized to its own maximum, so the POT
# scale cancels and either suite would give the same shape.
selected_true_q2 = nuwro["selected_true_q2"]
selected_true_q2_weights = nuwro["selected_true_q2_weights"]


## Axial form-factor prior comparison

Publication-styled version of the comparison in `zexp_FA_prior_builder.ipynb`. The NGEM `ZEXP_PRIORS` registry is the source of truth, so newly configured production priors are included automatically. Edit `SOURCE_VISIBILITY` in the next cell to choose which curves are drawn.

In [ ]:
import sys
from matplotlib.legend_handler import HandlerTuple

# Find uboone_ngem whether the notebook starts in ma_zexp/python or axial_mass.
search_root = Path.cwd().resolve()
ngem_repo = next(
    (parent / "uboone_ngem" for parent in (search_root, *search_root.parents)
     if (parent / "uboone_ngem/src/zexp_reweighting.py").is_file()),
    None,
)
if ngem_repo is None:
    raise FileNotFoundError("Could not locate uboone_ngem/src/zexp_reweighting.py")
if str(ngem_repo) not in sys.path:
    sys.path.insert(0, str(ngem_repo))

from src.zexp_reweighting import (
    AXIAL_FORM_FACTOR_Q2_ZERO, ZEXP_PRIORS, axial_form_factor_zexp,
    complete_zexp_a_values,
)

# Toggle sources and edit their displayed legend names here. The dictionary
# keys remain synchronized with NGEM; only the SOURCE_LABELS values are cosmetic.
SOURCE_VISIBILITY = {
    "Deuterium 2016": True,
    "MINERvA hydrogen 2023 kmax=8": False,
    "MINERvA hydrogen 2026 kmax=7": False,
    "MINERvA hydrogen 2026 kmax=6": True,
    "LQCD 2026 kmax=6": True,
    "MINERvA+LQCD 2026 kmax=6": True
}
SOURCE_LABELS = {
    "Deuterium 2016": r"Deuterium (2016) prior, $k_{\max}=8$",
    "MINERvA hydrogen 2023 kmax=8": r"MINERvA (2023) prior, $k_{\max}=8$",
    "MINERvA hydrogen 2026 kmax=6": r"MINERvA (2026) prior, $k_{\max}=6$",
    "MINERvA hydrogen 2026 kmax=7": r"MINERvA (2026) prior, $k_{\max}=7$",
    "LQCD 2026 kmax=6": r"LQCD (2026) prior, $k_{\max}=6$",
    "MINERvA+LQCD 2026 kmax=6": r"MINERvA + LQCD (2026) prior, $k_{\max}=6$",
}
SOURCE_PLOT_ORDER = [
    "Deuterium 2016",
    "MINERvA hydrogen 2023 kmax=8",
    "MINERvA hydrogen 2026 kmax=7",
    "MINERvA hydrogen 2026 kmax=6",
    "LQCD 2026 kmax=6",
    "MINERvA+LQCD 2026 kmax=6",
]
SOURCE_COLORS = {
    "Deuterium 2016": "#C000C3",                    # blue
    "MINERvA hydrogen 2023 kmax=8": "#E69F00",     # orange
    "MINERvA hydrogen 2026 kmax=7": "#D55E00",     # vermillion
    "MINERvA hydrogen 2026 kmax=6": "#56B4E9",     # sky blue
    "LQCD 2026 kmax=6": "#009E73",                 # green
    "MINERvA+LQCD 2026 kmax=6": "#CC79A7",         # purple
}

# Optional (minimum, maximum) Q^2 limits in GeV^2 for each displayed source.
# Use None for an unrestricted endpoint, e.g. (0.1, None) or (None, 2.0).
SOURCE_Q2_LIMITS_GEV2 = {prior.name: (None, None) for prior in ZEXP_PRIORS}
# SOURCE_Q2_LIMITS_GEV2["Deuterium 2016"] = (None, 1.0)

q2_ticks  = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]
q2_labels = ["0.01", "0.05", "0.1", "0.5", "1.0", "2.0"]
Q2_RANGE_GEV2 = (1.0e-2, 2.0)
N_PRIOR_DRAWS = 5000
RANDOM_SEED = 12345

def _dipole_fa(q2, ma_gev=1.014):
    return AXIAL_FORM_FACTOR_Q2_ZERO / (1.0 + q2 / ma_gev**2)**2

def _prior_curves(prior, q2, rng, n_draws):
    central = axial_form_factor_zexp(
        q2, prior.full_a_values, prior.t0_gev2, prior.t_cut_gev2
    )
    draws = rng.multivariate_normal(
        prior.free_a_values, prior.covariance, size=n_draws,
        check_valid="raise",
    )
    curves = np.empty((n_draws, len(q2)))
    for draw_index, free_values in enumerate(draws):
        full_values = complete_zexp_a_values(
            free_values, prior.kmax, prior.t0_gev2,
            t_cut_gev2=prior.t_cut_gev2, fa_q2_zero=prior.fa_q2_zero,
        )
        curves[draw_index] = axial_form_factor_zexp(
            q2, full_values, prior.t0_gev2, prior.t_cut_gev2
        )
    return central, np.percentile(curves, [16.0, 84.0], axis=0)

def plot_axial_form_factor_priors(
    source_visibility=SOURCE_VISIBILITY, source_labels=SOURCE_LABELS,
    source_q2_limits=SOURCE_Q2_LIMITS_GEV2,
):
    q2 = np.geomspace(*Q2_RANGE_GEV2, 360)
    reference = _dipole_fa(q2)
    dipole_ma_uncertainty_gev = 0.014
    dipole_variations = np.vstack([
        _dipole_fa(q2, 1.014 - dipole_ma_uncertainty_gev),
        _dipole_fa(q2, 1.014 + dipole_ma_uncertainty_gev),
    ])
    dipole_lower = np.min(dipole_variations, axis=0)
    dipole_upper = np.max(dipole_variations, axis=0)
    dipole_ratio_variations = dipole_variations / reference
    dipole_ratio_lower = np.min(dipole_ratio_variations, axis=0)
    dipole_ratio_upper = np.max(dipole_ratio_variations, axis=0)
    rng = np.random.default_rng(RANDOM_SEED)

    curve_sets = []
    if source_visibility.get("Deuterium 2016", False):
        deut_free = np.array([2.3, -0.6, -3.8, 2.3])
        deut_errors = np.sqrt([0.0154, 1.08, 6.54, 7.40])
        deut_corr = np.array([
            [1, .335, -.678, .611], [.350, 1, -.898, .367],
            [-.678, -.898, 1, -.685], [.611, .367, -.685, 1],
        ])
        deut_cov = np.outer(deut_errors, deut_errors) * (deut_corr + deut_corr.T) / 2
        eigenvalues, eigenvectors = np.linalg.eigh(deut_cov)
        deut_cov = (eigenvectors * np.clip(eigenvalues, 0, None)) @ eigenvectors.T
        deut_full = complete_zexp_a_values(
            deut_free, 8, -0.28, t_cut_gev2=9 * 0.139570**2,
            fa_q2_zero=AXIAL_FORM_FACTOR_Q2_ZERO,
        )
        deut_central = axial_form_factor_zexp(q2, deut_full, -0.28, 9 * 0.139570**2)
        deut_draws = rng.multivariate_normal(deut_free, deut_cov, size=N_PRIOR_DRAWS)
        deut_curves = np.array([
            axial_form_factor_zexp(
                q2, complete_zexp_a_values(
                    draw, 8, -0.28, t_cut_gev2=9 * 0.139570**2,
                    fa_q2_zero=AXIAL_FORM_FACTOR_Q2_ZERO,
                ), -0.28, 9 * 0.139570**2,
            ) for draw in deut_draws
        ])
        curve_sets.append(("Deuterium 2016", deut_central,
                           np.percentile(deut_curves, [16, 84], axis=0)))

    for prior in ZEXP_PRIORS:
        if source_visibility.get(prior.name, False):
            central, interval = _prior_curves(prior, q2, rng, N_PRIOR_DRAWS)
            curve_sets.append((prior.name, central, interval))

    plot_order = {name: index for index, name in enumerate(SOURCE_PLOT_ORDER)}
    curve_sets.sort(key=lambda curve: plot_order.get(curve[0], len(plot_order)))

    if not curve_sets:
        raise ValueError("Enable at least one entry in SOURCE_VISIBILITY")

    colors = plt.cm.tab10(np.linspace(0, 1, max(10, len(curve_sets))))
    linestyles = ["-", "--", "-.", ":"]
    with mpl.rc_context({"font.family": "sans-serif", "mathtext.fontset": "dejavusans"}):
        fig, (ax_fa, ax_ratio) = plt.subplots(
            2, 1, figsize=(7.0, 9.0), sharex=True, constrained_layout=True,
            gridspec_kw={"height_ratios": [1, 1.08]},
        )
        legend_handles = []
        legend_labels = []
        for source_index, (label, central, interval) in enumerate(curve_sets):
            color = SOURCE_COLORS.get(label, colors[source_index])
            linestyle = linestyles[(source_index // len(colors)) % len(linestyles)]
            display_label = source_labels.get(label, label)
            q2_min, q2_max = source_q2_limits.get(label, (None, None))
            source_mask = np.ones(q2.shape, dtype=bool)
            if q2_min is not None:
                source_mask &= q2 >= q2_min
            if q2_max is not None:
                source_mask &= q2 <= q2_max
            source_q2 = q2[source_mask]
            # The conventional axial form factor is negative. Plot its
            # negative so the magnitude is read naturally as positive.
            source_central = -central[source_mask]
            source_interval = -interval[::-1, source_mask]
            source_reference = -reference[source_mask]
            ax_fa.fill_between(
                source_q2, source_interval[0], source_interval[1],
                color=color, alpha=0.25, linewidth=0
            )
            ax_fa.plot(
                source_q2, source_central, color=color, linestyle=linestyle,
                linewidth=2.0,
            )
            legend_handles.append((
                mpl.patches.Patch(facecolor=color, edgecolor="none", alpha=0.25),
                mpl.lines.Line2D([], [], color=color, linestyle=linestyle, linewidth=2.0),
            ))
            legend_labels.append(display_label)

            ratio = source_central / source_reference
            lower, upper = source_interval / source_reference
            ax_ratio.fill_between(
                source_q2, lower, upper, color=color, alpha=0.25, linewidth=0
            )
            ax_ratio.plot(
                source_q2, ratio, color=color, linestyle=linestyle, linewidth=2.0
            )

        dipole_color = "#303642"
        ax_fa.fill_between(
            q2, -dipole_upper, -dipole_lower,
            facecolor=mpl.colors.to_rgba(dipole_color, 0.25),
            edgecolor=dipole_color, hatch="////", linewidth=0.35, zorder=3,
        )
        dipole_line, = ax_fa.plot(
            q2, -reference, color=dipole_color, linestyle=":", linewidth=1.4,
            zorder=4,
        )
        dipole_handle = (
            mpl.patches.Patch(
                facecolor=mpl.colors.to_rgba(dipole_color, 0.25),
                edgecolor=dipole_color, hatch="////", linewidth=0.35
            ),
            dipole_line,
        )

        # Arbitrary-normalized true-Q^2 silhouette for the selected MC.
        true_q2_edges = np.geomspace(*Q2_RANGE_GEV2, 20)
        true_q2_counts, _ = np.histogram(
            selected_true_q2, bins=true_q2_edges,
            weights=selected_true_q2_weights,
        )
        true_q2_shape = true_q2_counts / true_q2_counts.max()
        ax_true_q2 = ax_fa.twinx()
        ax_true_q2.stairs(
            true_q2_shape, true_q2_edges, fill=True,
            facecolor=mpl.colors.to_rgba("#6C757D", 0.35),
            edgecolor=mpl.colors.to_rgba("#4B535B", 0.50), linewidth=1.2,
            zorder=0,
        )
        ax_true_q2.set_ylim(0, 2.8)
        ax_true_q2.set_yticks([])
        ax_true_q2.spines["right"].set_visible(False)
        ax_true_q2.set_zorder(0)
        ax_fa.set_zorder(1)
        ax_fa.patch.set_alpha(0)
        true_q2_handle = mpl.patches.Patch(
            facecolor=mpl.colors.to_rgba("#6C757D", 0.25),
            edgecolor=mpl.colors.to_rgba("#4B535B", 0.40),
        )
        ax_fa.set_ylabel(r"$-F_A(Q^2)$")
        ax_ratio.legend(
            [*legend_handles, dipole_handle, true_q2_handle],
            [*legend_labels, r"Dipole, $M_A=1.014 \pm 0.014$ GeV",
             r"MicroBooNE expected true-$Q^2$ distribution"],
            handler_map={tuple: HandlerTuple(ndivide=1)},
            loc="upper left", fontsize=11, ncol=1, columnspacing=1.0, handlelength=2.5,
            frameon=False,
        )

        ax_ratio.fill_between(
            q2, dipole_ratio_lower, dipole_ratio_upper,
            facecolor=mpl.colors.to_rgba(dipole_color, 0.25),
            edgecolor=dipole_color, hatch="////", linewidth=0.35, zorder=3,
        )
        ax_ratio.axhline(
            1.0, color=dipole_color, linestyle=":", linewidth=1.1, zorder=4
        )
        ax_ratio.set(
            xscale="log", xlim=Q2_RANGE_GEV2, ylim=(0.5, 2),
            ylabel=r"$F_A(Q^2)/F_A^{\mathrm{dipole}}(Q^2; M_A=1.014\,\mathrm{GeV})$",
        )
        
        ax_ratio.set_xticks(
            q2_ticks, labels=q2_labels
        )
        for axis in (ax_fa, ax_ratio):
            axis.grid(which="major", color="#9AA4B2", alpha=0.22, linewidth=0.7)
            axis.grid(
                which="minor", axis="x", color="#9AA4B2", alpha=0.10,
                linewidth=0.5,
            )
        ax_ratio.text(
                0.01, 0.02, "Bands show 68% intervals", transform=ax_ratio.transAxes,
                ha="left", va="bottom", fontsize=12, color="#596273")
        fig.savefig(FIGURE_ROOT / "axial_form_factor_priors.pdf", dpi=600,
                    bbox_inches="tight", facecolor="white", format="pdf")
        plt.show()
    return fig, (ax_fa, ax_ratio)

fig_fa, (ax_fa, ax_fa_ratio) = plot_axial_form_factor_priors()

## Open data

The same two figures for the real beam-on sample, written with an `_opendata` suffix so the NuWro filenames the paper references are untouched. The prediction is unchanged; only the overlaid dataset, the POT normalization and the header label differ.


In [ ]:
opendata = prepare_suite(SUITES["opendata"])
plot_occupancy(opendata)
plot_slices(opendata)
plot_projections(opendata)
